# HRV-Based Physiological Reliability Analysis

This notebook demonstrates a basic signal processing pipeline for extracting HRV features from PPG signals.

The goal is to explore whether autonomic nervous system indicators derived from HRV can be used as physiological reliability signals for metabolic inference systems.


## Hypothesis

HRV features derived from PPG signals reflect autonomic nervous system states.

These physiological states may indicate conditions under which metabolic inference
becomes unreliable due to physiological decoupling.

Therefore, HRV can serve as a proxy signal for inference validity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("../figure", exist_ok=True)


plt.rcParams["figure.figsize"] = (12,4)

## Data Source and Physiological Proxy

PPG (Photoplethysmography) signals capture blood volume changes and
provide an accessible way to derive heart rate variability (HRV).

HRV reflects autonomic nervous system dynamics, including the balance
between sympathetic and parasympathetic activity.

Since autonomic states (e.g., stress) may modulate metabolic processes,
HRV is used here as a proxy for physiological context.


In [ ]:
file_path = "../date/extracted_data/bidmc/01/01_ppg.txt"

ppg = pd.read_csv(file_path, header=None)

signal = ppg[0].values

print("Signal length:", len(signal))


In [ ]:
plt.figure()

plt.plot(signal)

plt.title("Raw PPG Signal")

plt.xlabel("Time")

plt.ylabel("Amplitude")
plt.savefig("../figure/ppg_raw.png", dpi=300)


plt.show()

## Signal Preprocessing: Smoothing

Raw PPG signals contain high-frequency noise that can interfere with peak detection.

Accurate peak detection is critical because RR intervals are derived from peak locations,
and HRV features are computed from these intervals.

A moving average filter is applied to reduce noise while preserving the main waveform.

The window size (20 samples) is chosen as a simple initial approximation,
balancing noise reduction and signal distortion.


In [ ]:
ppg_smooth = pd.Series(signal).rolling(window=20).mean()

plt.figure()

plt.plot(ppg_smooth)

plt.title("Smoothed PPG Signal")

plt.xlabel("Time")

plt.ylabel("Amplitude")
plt.savefig("../figure/ppg_smooth.png", dpi=300)

plt.show()

## Peak Detection and Cardiac Cycle Approximation

Each detected peak in the PPG signal corresponds approximately to a heartbeat.

The time difference between successive peaks defines the RR interval,
which is the fundamental unit for HRV analysis.

Accurate peak detection is therefore essential for capturing physiological dynamics.


In [ ]:
peaks = []

for i in range(1, len(ppg_smooth)-1):
    
    if ppg_smooth[i] > ppg_smooth[i-1] and ppg_smooth[i] > ppg_smooth[i+1]:
        
        peaks.append(i)

peaks = np.array(peaks)

print("Number of peaks:", len(peaks))


In [ ]:
plt.figure()

plt.plot(ppg_smooth)

plt.scatter(peaks, ppg_smooth.iloc[peaks], color="red", s=10)

plt.title("Detected Peaks")

plt.xlabel("Time")

plt.ylabel("Amplitude")
plt.savefig("../figure/ppg_peaks.png", dpi=300)

plt.show()


## RR Interval Time Series

The RR interval sequence reflects beat-to-beat variability in heart rate.

Variations in RR intervals are influenced by autonomic nervous system activity.

Rather than being constant, these intervals evolve over time,
indicating that physiological state is dynamic rather than static.


In [ ]:
rr_intervals = np.diff(peaks)

plt.figure()

plt.plot(rr_intervals)

plt.title("RR Interval Time Series")

plt.xlabel("Beat Index")

plt.ylabel("Interval")
plt.savefig("../figure/RR_interval_time_series.png", dpi=300)

plt.show()

## HRV Feature Extraction and Physiological Interpretation

Two standard HRV features are computed:

- SDNN: reflects overall variability in heart rate
- RMSSD: reflects short-term variability and is strongly associated with parasympathetic activity

RMSSD is particularly relevant because parasympathetic activity tends to decrease
during acute stress and sympathetic activation.

Such physiological perturbations may introduce decoupling between metabolic signals
(e.g., VOCs) and underlying glucose dynamics.

Therefore, RMSSD is considered a candidate indicator of inference validity.


In [ ]:
sdnn = np.std(rr_intervals)

rmssd = np.sqrt(np.mean(np.diff(rr_intervals)**2))

print("SDNN:", sdnn)

print("RMSSD:", rmssd)

## Time-Varying HRV Analysis

Single summary statistics (e.g., one RMSSD value) cannot capture dynamic physiological changes.

To analyze temporal variation, RMSSD is computed over a sliding window.

This allows visualization of how autonomic state evolves over time,
which is critical for identifying transient conditions where inference may become unreliable.


In [ ]:
window = 50
rolling_rmssd = []

for i in range(len(rr_intervals) - window):
    segment = rr_intervals[i:i+window]
    value = np.sqrt(np.mean(np.diff(segment)**2))
    rolling_rmssd.append(value)

rolling_rmssd = np.array(rolling_rmssd)

plt.figure()
plt.plot(rolling_rmssd)
plt.title("Rolling RMSSD (Physiological State Over Time)")
plt.xlabel("Time Window")
plt.ylabel("RMSSD")
plt.savefig("../figure/rolling_rmssd.png", dpi=300)
plt.show()


## Prototype Reliability Gating

As a simple proof-of-concept, a threshold is applied to RMSSD
to distinguish between relatively stable and potentially unreliable physiological states.

Lower RMSSD values may indicate stress or autonomic imbalance,
which could correspond to conditions where metabolic inference becomes less reliable.

This demonstrates a minimal form of reliability-aware decision filtering.


In [ ]:
threshold = np.percentile(rolling_rmssd, 25)
reliability_flag = rolling_rmssd > threshold

plt.figure()
plt.plot(rolling_rmssd, label="RMSSD")
plt.axhline(threshold, linestyle="--", label="Threshold")
plt.title("Reliability Gating based on RMSSD")
plt.legend()
plt.savefig("../figure/reliability_gating.png", dpi=300)
plt.show()


In [ ]:
plt.figure()

plt.hist(rr_intervals, bins=30)

plt.title("RR Interval Distribution")

plt.xlabel("Interval")

plt.ylabel("Frequency")
plt.savefig("../figure/RR_interval_distribution.png", dpi=300)

plt.show()


In [ ]:
plt.figure()

plt.scatter(range(len(rr_intervals)), rr_intervals, s=5)

plt.title("RR Interval Scatter")

plt.xlabel("Beat Index")

plt.ylabel("RR Interval")
plt.savefig("../figure/RR_interval_scatter.png", dpi=300)

plt.show()

## Conclusion

This analysis demonstrates that HRV features derived from PPG signals
capture dynamic autonomic nervous system activity.

The results support the hypothesis that physiological state indicators,
such as RMSSD, can serve as proxy signals for inference validity.

In particular, temporal variation in HRV suggests that metabolic inference
may not be uniformly reliable, but instead depends on underlying physiological conditions.

This provides initial support for a reliability-aware framework,
where predictions are conditioned on physiological context
rather than assumed to be universally valid.
